In [3]:
import warnings
warnings.filterwarnings("ignore")

In [4]:
import scanpy as sc
import pandas as pd
import numpy as np
import torch
import dgl
import random
from scipy.sparse import csr_matrix
from sklearn.metrics import adjusted_rand_score as ari_score
from sklearn.metrics.cluster import normalized_mutual_info_score
import os

In [5]:
import so as so

In [6]:
seed = 42
random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
dgl.random.seed(seed)

In [7]:
sc.set_figure_params(dpi=150, figsize=(2, 2), frameon=False)    # TODO 是否画边框

In [8]:
import os

In [9]:
files = os.listdir('./')
files

['3_metircs_compare_all_methods.ipynb',
 'log',
 '1_Merfish_heart_train.ipynb',
 'all_methods_results',
 '.ipynb_checkpoints',
 '2_Merfish_heart_evaluate.ipynb']

In [10]:
adata_raw = sc.read_h5ad('/dataset/1_Merfish_human_heart/human_heart_pc13_merfish.h5ad')
adata_raw

AnnData object with n_obs × n_vars = 228635 × 238
    obs: 'sample_id', 'batch', 'n_counts', 'leiden', 'zone_cluster', 'communities', 'complexity', 'populations', 'purity', 'region', 'subregion', 'region5', 'region4'
    uns: 'communities_colors', 'leiden', 'leiden_colors', 'neighbors', 'pca', 'populations_colors', 'region4_colors', 'region5_colors', 'sample_id_colors', 'subregion_colors', 'umap', 'zone_cluster_colors'
    obsm: 'X_pca', 'X_umap', 'spatial'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

In [11]:
adata_raw.X.max()

1456

In [12]:
adata_raw.obs

,sample_id,batch,n_counts,leiden,zone_cluster,communities,complexity,populations,purity,region,subregion,region5,region4
6-R77_4C4,R77_4C4,R77_4C4,86.0,6,4,AVN/AV Ring,8,VIC,0.544534,AV_Valve,Atrioventricular Node / Atrioventricular Ring ...,Atrioventricular_ValveRegion,AV_Valve
8-R77_4C4,R77_4C4,R77_4C4,148.0,6,1,Valve,8,VIC,0.625984,AV_Valve,Valve Community,Atrioventricular_ValveRegion,AV_Valve
9-R77_4C4,R77_4C4,R77_4C4,100.0,6,1,Valve,8,VIC,0.583665,AV_Valve,Valve Community,Atrioventricular_ValveRegion,AV_Valve
10-R77_4C4,R77_4C4,R77_4C4,70.0,6,1,Valve,9,VIC,0.766393,AV_Valve,Valve Community,Atrioventricular_ValveRegion,AV_Valve
12-R77_4C4,R77_4C4,R77_4C4,63.0,6,1,Valve,8,VIC,0.596838,AV_Valve,Valve Community,Atrioventricular_ValveRegion,AV_Valve
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6660345-R78_4C15,R78_4C15,R78_4C15,70.0,6,1,Valve,9,VIC,0.724675,AV_Valve,Valve Community,Atrioventricular_ValveRegion,AV_Valve
6660348-R78_4C15,R78_4C15,R78_4C15,83.0,24,1,Valve,8,VEC,0.747340,AV_Valve,Valve Community,Atrioventricular_ValveRegion,AV_Valve
6660352-R78_4C15,R78_4C15,R78_4C15,70.0,24,1,Valve,7,VEC,0.771772,AV_Valve,Valve Community,Atrioventricular_ValveRegion,AV_Valve
6660357-R78_4C15,R78_4C15,R78_4C15,118.0,6,1,Valve,9,VIC,0.657963,AV_Valve,Valve Community,Atrioventricular_ValveRegion,AV_Valve


In [14]:
adata_raw.obs.loc[:, 'batch'].value_counts()

batch
R78_4C15    79891
R78_4C12    75782
R77_4C4     72962
Name: count, dtype: int64

In [15]:
len(adata_raw.obs.loc[:, 'batch'].unique())

3

In [16]:
data_list = [adata_raw[adata_raw.obs.loc[:, 'batch'] == batch, :] for batch in adata_raw.obs.loc[:, 'batch'].unique()]
data_list

[View of AnnData object with n_obs × n_vars = 72962 × 238
     obs: 'sample_id', 'batch', 'n_counts', 'leiden', 'zone_cluster', 'communities', 'complexity', 'populations', 'purity', 'region', 'subregion', 'region5', 'region4'
     uns: 'communities_colors', 'leiden', 'leiden_colors', 'neighbors', 'pca', 'populations_colors', 'region4_colors', 'region5_colors', 'sample_id_colors', 'subregion_colors', 'umap', 'zone_cluster_colors'
     obsm: 'X_pca', 'X_umap', 'spatial'
     varm: 'PCs'
     obsp: 'connectivities', 'distances',
 View of AnnData object with n_obs × n_vars = 75782 × 238
     obs: 'sample_id', 'batch', 'n_counts', 'leiden', 'zone_cluster', 'communities', 'complexity', 'populations', 'purity', 'region', 'subregion', 'region5', 'region4'
     uns: 'communities_colors', 'leiden', 'leiden_colors', 'neighbors', 'pca', 'populations_colors', 'region4_colors', 'region5_colors', 'sample_id_colors', 'subregion_colors', 'umap', 'zone_cluster_colors'
     obsm: 'X_pca', 'X_umap', 'spat

In [17]:
data_list[0].X.max(), data_list[0].X.min()

(array(679), array(0))

In [18]:
batch_names = adata_raw.obs.loc[:, 'batch'].unique()
# for i, adata in enumerate(data_list):
#     batch_names.append(f'adata{i}')
    # adata.obsm['spatial'] = adata.obs.loc[:, ['X', 'Y']].to_numpy()
batch_names

['R77_4C4', 'R78_4C12', 'R78_4C15']
Categories (3, object): ['R77_4C4', 'R78_4C12', 'R78_4C15']

In [19]:
batch_key = 'batch'
adata = so.pp.process_adata(data_list, label=batch_key, keys=batch_names, n_top_features=3000, target_sum=1e4)
adata

AnnData object with n_obs × n_vars = 228635 × 238
    obs: 'sample_id', 'batch', 'n_counts', 'leiden', 'zone_cluster', 'communities', 'complexity', 'populations', 'purity', 'region', 'subregion', 'region5', 'region4', 'original_index', 'spot_quality'
    var: 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'highly_variable_nbatches'
    uns: 'log1p', 'mu_bg', 'std_bg'
    obsm: 'X_pca', 'X_umap', 'spatial'
    layers: 'counts', 'norm_log'

In [20]:
adata.X.max(), adata.X.min()

(8.899041, 0.0)

In [21]:
rad_cutoff_list = [40] * len(data_list)
adata, g = so.pp.process_graph(adata, data_list, cutoff_list=rad_cutoff_list)

cal spatial net in data_list
------Calculating spatial graph...
The graph contains 1936642 edges, 72962 cells.
26.5432 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 1986222 edges, 75782 cells.
26.2097 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 2095970 edges, 79891 cells.
26.2354 neighbors per cell on average.


In [22]:
path_results = f'./log/integration_radius30/'

In [23]:
adata_int = so.model.SpaVCCA(adata=adata,
                            n_epoch=5,
                            batch_size=1024,
                            verbose=False,
                            gpu=0,
                            random_seed=42,
                            output_dir=path_results,
                            num_heads=2,
                            h_dim=16,
                            reconstruct_loss='orig',
                            alpha_kl=1,
                            alpha_re=1,
                            alpha_mmd=1,
                            alpha_spl=1,
                            )

clip
reconstruct_loss == orig


Epochs: 100%|█| 5/5 [00:54<00:00, 10.88s/it, kl_loss=5.4159, recon_loss=522.0802, cca_loss=0.0025, sub_par_loss=0.3561, 
Infer latent: 100%|███████████████████████████████████████████████████████████████████| 224/224 [00:02<00:00, 90.62it/s]


In [25]:
del adata_int.uns['adj']

In [26]:
adata_int.write_h5ad(f'{path_results}adata_int.h5ad', compression='gzip')